# Load depedencies

## Install depedencies

In [1]:
%%capture
%pip install -U langchain langchainhub langchain_community langchain-huggingface
%pip install -U faiss-gpu transformers accelerate

In [2]:
!pip install -U sentence-transformers langchain-huggingface
!pip uninstall -y torchaudio
!pip install openai
%pip install -U langchain-text-splitters
!pip install jq
!pip install packaging==24.2
!pip install -U sentence-transformers


Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip install openai==1.35.0 aiohttp>=3.10.0


In [4]:
import sys
# 强制拦截torchaudio模块加载
sys.modules["torchaudio"] = None
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)



## Import depedencies

In [5]:
%%time
import torch

from langchain_community.document_loaders import JSONLoader
from langchain_core.prompts import PromptTemplate

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from transformers import GenerationConfig
import langchainhub as hub
from IPython.display import clear_output
import os
from openai import OpenAI

clear_output()


CPU times: user 8.75 s, sys: 1.67 s, total: 10.4 s
Wall time: 10.6 s


## Hugging Face login

In [6]:
# from huggingface_hub import login
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()

# hf_token = user_secrets.get_secret("HUGGINGFACE_TOKEN")
# login(token = hf_token)

# Load Dataset

In [7]:
import json
from langchain_core.documents import Document
# ✅ 新版正确导入
from langchain_text_splitters import RecursiveCharacterTextSplitter


file_path = "/kaggle/input/datasets/ysqsww/ymbbkp/data.jsonl"
raw_docs = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        data = json.loads(line)
        doc = Document(page_content=data["text"])
        raw_docs.append(doc)

# 新增：文本切分器（原代码缺失）
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    length_function=len
)
split_docs = text_splitter.split_documents(raw_docs)
print(f"文档块总数：{len(split_docs)}")


文档块总数：11


# Load Embedding model

In [8]:
# Define the path to the pre-trained model you want to use
# modelPath = "sentence-transformers/all-MiniLM-L12-v2"
modelPath = "BAAI/bge-large-zh-v1.5"

# Create a dictionary with model configuration options, specifying to use the GPU for computations
model_kwargs = {'device':'cuda',"trust_remote_code": True}

# Create a dictionary with encoding options, specifically setting 'normalize_embeddings' to False
encode_kwargs = {'normalize_embeddings': False}

# Initialize an instance of HuggingFaceEmbeddings with the specified parameters
embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,     
    model_kwargs=model_kwargs, 
    encode_kwargs=encode_kwargs
)

/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use th

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use th

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use th

...

tokenizer.json:   0%|          | 0.00/439k [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use th

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use th

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [9]:
text = "你好啊月亮"
query_result = embeddings.embed_query(text)
query_result[:3]

[0.0767059400677681, -0.0046313065104186535, -0.043977461755275726]

## Create a VectorDB

In [10]:
vector_db = FAISS.from_documents(split_docs, embeddings)
vector_db.save_local("/kaggle/working/faiss_doctor_index")


In [11]:
# help(vector_db)

In [12]:
question = "三原色是什么？"
searchDocs = vector_db.similarity_search(question)
x = searchDocs[0]


In [13]:
x

Document(id='325a85b9-5632-4d78-8ee1-8fc687d4aff2', metadata={}, page_content='| 职级 | 头衔 | 典型岗位 | 地球年薪 | 月球补贴 | 火星补贴 |\n|------|------|---------|---------|---------|---------|\n| P1 | 见习专员 | 应届生 | 15万 | +8万 | +15万 |\n| P2 | 初级专员 | 1-3年经验 | 25万 | +12万 | +22万 |\n| P3 | 中级专员 | 3-5年经验 | 40万 | +18万 | +35万 |\n| P4 | 高级专员 | 5-8年经验 | 60万 | +25万 | +50万 |\n| P5 | 资深专家 | 8年以上 | 90万 | +35万 | +70万 |\n| P6 | 首席专家 | 行业顶尖 | 150万 | +50万 | +100万 |\n| M1 | 团队主管 | 管理5-10人 | 70万 | +28万 | +55万 |\n| M2 | 部门经理 | 管理20-50人 | 120万 | +40万 | +80万 |\n| M3 | 事业部总监 | 管理100人以上 | 200万 | +60万 | +120万 |')

In [14]:
s = x.page_content
print(s.encode().decode("unicode_escape"))

| èçº§ | å¤´è¡ | å¸åå²ä½ | å°çå¹´èª | æçè¡¥è´´ | ç«æè¡¥è´´ |
|------|------|---------|---------|---------|---------|
| P1 | è§ä¹ ä¸å | åºå±ç | 15ä¸ | +8ä¸ | +15ä¸ |
| P2 | åçº§ä¸å | 1-3å¹´ç»éª | 25ä¸ | +12ä¸ | +22ä¸ |
| P3 | ä¸­çº§ä¸å | 3-5å¹´ç»éª | 40ä¸ | +18ä¸ | +35ä¸ |
| P4 | é«çº§ä¸å | 5-8å¹´ç»éª | 60ä¸ | +25ä¸ | +50ä¸ |
| P5 | èµæ·±ä¸å®¶ | 8å¹´ä»¥ä¸ | 90ä¸ | +35ä¸ | +70ä¸ |
| P6 | é¦å¸­ä¸å®¶ | è¡ä¸é¡¶å° | 150ä¸ | +50ä¸ | +100ä¸ |
| M1 | å¢éä¸»ç®¡ | ç®¡ç5-10äºº | 70ä¸ | +28ä¸ | +55ä¸ |
| M2 | é¨é¨ç»ç | ç®¡ç20-50äºº | 120ä¸ | +40ä¸ | +80ä¸ |
| M3 | äºä¸é¨æ»ç | ç®¡ç100äººä»¥ä¸ | 200ä¸ | +60ä¸ | +120ä¸ |


# Load Model

In [15]:
base_model = "/kaggle/input/qwen2/transformers/qwen2-7b-instruct/1"

tokenizer = AutoTokenizer.from_pretrained(base_model)

model = AutoModelForCausalLM.from_pretrained(
        base_model,
        return_dict=True,
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: The `ipykernel.comm.Comm` class has been deprecated. Please use the `comm` module instead.For creating comms, use the function `from comm import create_comm`.
  self.comm = Comm(**args)
/opt/conda/lib/python3.10/site-packages/ipywidgets/widgets/widget.py:503: DeprecationWarning: T

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## Create Pipeline

In [16]:
gen_config = GenerationConfig(
    max_new_tokens=1024
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    generation_config=gen_config,
    return_full_text=False,
    clean_up_tokenization_spaces=False
)
llm = HuggingFacePipeline(pipeline=pipe)

# Create chain

In [17]:
retriever = vector_db.as_retriever(search_kwargs={"k": 3})
from langchain_core.prompts import PromptTemplate
rag_prompt = PromptTemplate.from_template ("""
你是基于员工手册回答问题的助手，只能使用下面检索到的上下文信息回答。
如果上下文没有相关信息，直接回答【知识库中没有相关内容】，严禁编造信息。
上下文：{context}
问题：{question}
Answer:
""")
qa_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

In [18]:
# help(vector_db)

## Testing

In [19]:
question_1 = "星际桥公司的CEO是谁？"
result_1 = qa_chain.invoke(question_1)
parts = result_1.split("Answer: ")
if len(parts)>=2:
    print(parts[1])
else:
    print(result_1)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


星际桥公司的CEO是陈星桥。他同时也是公司的创始人，以前是一名NASA工程师。


In [20]:
question_2 = "去火星的单程经济舱票价是多少？"
result_2 = qa_chain.invoke(question_2)
parts = result_2.split("Answer: ")
if len(parts)>=2:
    print(parts[1])
else:
    print(result_2)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


去火星的单程经济舱票价是120万地球币。


In [21]:
question_3 = "月球基地的常驻人数是多少？"
result_3 = qa_chain.invoke(question_3)
parts = result_3.split("Answer: ")
if len(parts)>=2:
    print(parts[1])
else:
    print(result_3)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


月球基地（静海区）的常驻人数是600人。


In [22]:
question_4 = "火星基地的特色餐有什么？"
result_4 = qa_chain.invoke(question_4)
parts = result_4.split("Answer: ")
if len(parts)>=2:
    print(parts[1])
else:
    print(result_4)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


火星基地的特色餐包括"红土土豆泥"和"火星蓝莓"。


In [23]:
question_5 = "员工每年有几次免费地球-火星往返额度？"
result_5 = qa_chain.invoke(question_5)
parts = result_5.split("Answer: ")
if len(parts)>=2:
    print(parts[1])
else:
    print(result_5)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


员工每年有1次免费地球-火星往返额度（仅限P4及以上职级）。


In [24]:
question_6 = "在静默日说话会有什么惩罚？"
result_6 = qa_chain.invoke(question_6)
parts = result_6.split("Answer: ")
if len(parts)>=2:
    print(parts[1])
else:
    print(result_6)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


知识库中没有相关内容


In [25]:
question_7 = "星际桥公司有多少女员工？"
result_7 = qa_chain.invoke(question_7)
parts = result_7.split("Answer: ")
if len(parts)>=2:
    print(parts[1])
else:
    print(result_7)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


知识库中没有相关内容


In [26]:
question_8 = "星际桥公司去年的营收是多少？"
result_8 = qa_chain.invoke(question_8)
parts = result_8.split("Answer: ")
if len(parts)>=2:
    print(parts[1])
else:
    print(result_8)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


知识库中没有相关内容
